## 3_reweighting_examples.ipynb
### Fairness and Compliance of AI
### University of Oklahoma

This notebook illustrates techniques for addressing imbalance in a dataset. In particular, it focuses on addressing imbalance with respect to sensitive attributes.  The notebook uses the UCI Adult Census dataset. It starts with the CSV file from imputation_examples.ipynb:  _adult_imputed.csv_

The notebook includes methods for addressing imbalance in both classification and regression. Inverse Prevalence Reweighting can be used for both classification and regression, but we only use it on classification here:
1. Inverse Prevalence Reweighting (classification)
2. Kamiran-Calders Reweighting (classification)
3. Residual-Parity Reweighting (regression)

Throughout this notebook we set explicit random seeds to ensure identical results across runs and machines.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, \
     mean_absolute_percentage_error, mean_squared_error
import os, random

# Set global seeds for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
random.seed(42)
np.random.seed(42)


### Data preparation
Load the imputed adult dataset and prepare features and labels for regression modeling. We will predict the income levels for each observation via classification using logistic regression. That means that we have to encode the categorical income variable, 'target', as an integer (in this case 0/1). We then one-hot encode categorical variables in the predictor set. Finally, we z-score normalize numerical predictors. We conclude by putting the predictor data back into a single dataset. 

In [ ]:
# ---------------------------------------------------------
# 1. CLASSIFICATION REWEIGHTING
# ---------------------------------------------------------

# ---------------------------------------------------------
# 1.1. Load Adult dataset and encode labels
# ---------------------------------------------------------
df = pd.read_csv("adult_imputed.csv")

# Protected attribute (A) and label (Y)
A = df["sex"]
Y_class = df["target"]

# Encode protected attribute and label for weighting + evaluation
A_le = LabelEncoder()
Y_le = LabelEncoder()
A_enc = A_le.fit_transform(A)
Y_class_enc = Y_le.fit_transform(Y_class)

# ---------------------------------------------------------
# Build predictor matrix and transform
# ---------------------------------------------------------
X = df.drop(columns=["target"])   # <-- sex stays in the predictors

# Split numeric and categorical
numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(exclude=[np.number]).columns

# Z-score normalize numeric predictors
scaler = StandardScaler()
X_numeric_scaled = pd.DataFrame(
    scaler.fit_transform(X[numeric_cols]),
    columns=numeric_cols,
    index=X.index
)

# One-hot encode categorical predictors (including 'sex')
X_categorical = pd.get_dummies(X[categorical_cols], drop_first=True)

# Combine into final feature matrix
X_enc = pd.concat([X_numeric_scaled, X_categorical], axis=1)


### Reweighting for Classification
Calculate the weights using two different methods, Inverse Prevalence Reweighting and Kamiran-Calders Reweighting. The weights are based upon the combinations of the protected attribute _sex_, (A) and the target income variable _target_, (Y).

In [ ]:
# ---------------------------------------------------------
# 1.2. Calculate fairness weights
# ---------------------------------------------------------
df_class = pd.DataFrame({"A": A_enc, "Y": Y_class_enc})

counts = df_class.groupby(["A", "Y"]).size()

def inv_prev_weight(a, y):
    """
    Inverse Prevalence Reweighting
    """
    return 1.0 / counts[a, y]

weights_inv_class = df_class.apply(lambda r: inv_prev_weight(r["A"], r["Y"]), axis=1)

P_A = df_class["A"].value_counts(normalize=True)
P_Y = df_class["Y"].value_counts(normalize=True)
P_AY = df_class.groupby(["A", "Y"]).size() / len(df_class)

def kc_weight(a, y):
    """
    Kamiran–Calders Reweighting
    """
    return (P_A[a] * P_Y[y]) / P_AY[a, y]

weights_kc = df_class.apply(lambda r: kc_weight(r["A"], r["Y"]), axis=1)


### Run Classification
Run classification by logistic regression using the unweighted and two weighted datasets.

In [ ]:
# ---------------------------------------------------------
# 1.3. Train classification models, run, and view results
# ---------------------------------------------------------
X_train, X_test, \
y_train, y_test, \
w_inv_train, w_inv_test, \
w_kc_train, w_kc_test = train_test_split(
    X_enc, Y_class_enc, weights_inv_class, weights_kc,
    test_size=0.3, random_state=42
)

clf_unweighted = LogisticRegression(max_iter=500, random_state=42)
clf_unweighted.fit(X_train, y_train)

clf_inv = LogisticRegression(max_iter=500, random_state=42)
clf_inv.fit(X_train, y_train, sample_weight=w_inv_train)

clf_kc = LogisticRegression(max_iter=500, random_state=42)
clf_kc.fit(X_train, y_train, sample_weight=w_kc_train)

print("\n=== Classification: Unweighted ===")
print(classification_report(y_test, clf_unweighted.predict(X_test)))

print("\n=== Classification: Inverse Prevalence Reweighting ===")
print(classification_report(y_test, clf_inv.predict(X_test)))


### Test classification for bias
Use the error-rate parity test to check for separation.  We want FPR and FNR for females and males to be close to equivalent.

In [ ]:
# ---------------------------------------------------------
# 1.4. Classification Error-Rate Parity Test (Separation)
# ---------------------------------------------------------

def error_rates_by_group(y_true, y_pred, A_test, A_encoder, Y_encoder):
    """
    Calculate False Positive Rate and False Negative Rate by protected group. This function
    uses the encoded protected attribute values and label values, but decodes them in the output
    for readability.

    parameters:
    - y_true: true labels
    - y_pred: predicted labels
    - A_test: protected attribute values for test set
    - A_encoder: LabelEncoder for protected attribute (to decode group names)
    - Y_encoder: LabelEncoder for label (to decode label names)

    returns:
    - DataFrame with FPR and FNR for each protected group
    """
    
    results = []

    for group in np.unique(A_test):
        mask = (A_test == group)
        yt = y_true[mask]
        yp = y_pred[mask]

        # False Positive Rate: FP / (FP + TN)
        fp = np.sum((yp == 1) & (yt == 0))
        tn = np.sum((yp == 0) & (yt == 0))
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        # False Negative Rate: FN / (FN + TP)
        fn = np.sum((yp == 0) & (yt == 1))
        tp = np.sum((yp == 1) & (yt == 1))
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan

        results.append({
            "sex": A_encoder.inverse_transform([group])[0],
            "FPR (predicting {})".format(Y_encoder.inverse_transform([1])[0]): round(fpr, 3),
            "FNR (predicting {})".format(Y_encoder.inverse_transform([1])[0]): round(fnr, 3)
        })

    return pd.DataFrame(results)


# Extract protected attribute for the test split
_, A_test = train_test_split(A_enc, test_size=0.3, random_state=42)

pred_unweighted = clf_unweighted.predict(X_test)
pred_inv = clf_inv.predict(X_test)
pred_kc = clf_kc.predict(X_test)

print("\n=== Error-Rate Parity: Unweighted Model ===")
print(error_rates_by_group(y_test, pred_unweighted, A_test, A_le, Y_le))

print("\n=== Error-Rate Parity: Inverse Prevalence Reweighting ===")
print(error_rates_by_group(y_test, pred_inv, A_test, A_le, Y_le))

print("\n=== Error-Rate Parity: Kamiran–Calders Reweighting ===")
print(error_rates_by_group(y_test, pred_kc, A_test, A_le, Y_le))



### Regression Reweighting
Weighting for regression is different. Here we will run regression to determine years of education based upon all of the other features with the exception of _target_. We first run a baseline regression. Then we use the residuals by protected attribute to determine new weights for observations. We then run the regression again. We begin by modifying the input dataframe.

In [ ]:
# ---------------------------------------------------------
# 2. REGRESSION REWEIGHTING (CLEAN + FIXED)
# ---------------------------------------------------------

# ---------------------------------------------------------
# 2.1 Prepare dataframe
# ---------------------------------------------------------

# Start from original df and drop the classification target
df_reg = df.drop(columns=["target"])

# Encode the protected attribute for later use in weighting and evaluation
A = df_reg["sex"]
A_le = LabelEncoder()
A_enc = A_le.fit_transform(A)

# Regression target (num_edu) — also z-score normalize if desired
Y_reg_raw = df_reg["num_edu"].astype(float)

num_edu_scaler = StandardScaler()
#Y_reg = num_edu_scaler.fit_transform(Y_reg_raw.values.reshape(-1, 1)).flatten() # Normalized target if wanted
Y_reg = Y_reg_raw  # Uses raw target for interpretability of fairness metrics

# Features: drop regression target
X = df_reg.drop(columns=["num_edu"])

# ---------------------------------------------------------
# 2.2 Normalize numeric predictors + one-hot encode categoricals
# ---------------------------------------------------------

numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(exclude=[np.number]).columns

# Z-score numeric predictors
scaler = StandardScaler()
X_numeric_scaled = pd.DataFrame(
    scaler.fit_transform(X[numeric_cols]),
    columns=numeric_cols,
    index=X.index
)

# One-hot encode categoricals
X_categorical = pd.get_dummies(X[categorical_cols], drop_first=True)

# Final encoded feature matrix
X_enc = pd.concat([X_numeric_scaled, X_categorical], axis=1)



### Calculate fairness weights
Residual-parity reweighting specifically seeks to address differential performance across protected attribute values by using regression residuals to more heavily weight the class with lower accuracy in subsequent regressions.

In [ ]:
# ---------------------------------------------------------
# 2.3 Residual-Parity Reweighting
# ---------------------------------------------------------

X_train_r, X_test_r, y_train_r, y_test_r, A_train_r, A_test_r = train_test_split(
    X_enc, Y_reg, A_enc, test_size=0.3, random_state=42
)

# Baseline model
baseline = LinearRegression()
baseline.fit(X_train_r, y_train_r)
y_pred_r = baseline.predict(X_train_r)

# Compute residuals
residuals = y_train_r - y_pred_r

# Compute mean absolute residual per group
group_residuals = {
    a: np.mean(np.abs(residuals[A_train_r == a]))
    for a in np.unique(A_train_r)
}

def residual_parity_weight(a, r):
    """
    Residual-Parity Reweighting: weight is proportional to how large the residual is for that example,
    normalized by the average residual for that group. Examples with larger errors get more weight,
    and groups with larger average errors also get more weight overall.

    parameters:
    - a: protected attribute value for the example
    - r: residual for the example (y_true - y_pred)

    returns:
    - weight for the example
    """
    # Larger residual → larger weight
    return np.abs(r) / (group_residuals[a] + 1e-8)

# Apply residual-parity weights to each example in the training set
weights_resid = np.array([
    residual_parity_weight(a, r)
    for a, r in zip(A_train_r, residuals)
])


### Run the weighted regression and generate basic results

In [ ]:
# ---------------------------------------------------------
# 2.4 Run the weighted regression model and evaluate
# ---------------------------------------------------------

# Weighted regression model
rsd_weighted = LinearRegression()
rsd_weighted.fit(X_train_r, y_train_r, sample_weight=weights_resid)

print("\n=== Regression: Unweighted MSE ===")
print(mean_squared_error(y_test_r, baseline.predict(X_test_r)))

print("\n=== Regression: Residual-Parity Weighted MSE ===")
print(mean_squared_error(y_test_r, rsd_weighted.predict(X_test_r)))


### Compare performance across protected attributes for regression methods
We compare mean average percent error between the protected attribute classes.  We hope that the weighted regression reduces the gap between MAPEs of the classes.

In [ ]:
# ---------------------------------------------------------
# 2.5 Fairness Evaluation (Decoded Output)
# ---------------------------------------------------------

def regression_mape_by_group(y_true, y_pred, A_test, A_encoder):
    results = []

    for group in np.unique(A_test):
        mask = (A_test == group)
        yt = y_true[mask]
        yp = y_pred[mask]

        # Avoid division-by-zero in MAPE
        nonzero_mask = yt != 0
        if np.sum(nonzero_mask) == 0:
            mape = np.nan
        else:
            mape = np.mean(np.abs((yt[nonzero_mask] - yp[nonzero_mask]) / yt[nonzero_mask])) * 100

        # Mean Signed Error (systematic over/under prediction)
        bias = np.mean(yp - yt)

        results.append({
            "sex": A_encoder.inverse_transform([group])[0],
            "MAPE (%)": round(mape, 6),
            "Bias (yp - yt)": round(bias, 6)
        })

    return pd.DataFrame(results)


print("\n=== Fairness Evaluation: Baseline Regression ===")
print(regression_mape_by_group(y_test_r, baseline.predict(X_test_r), A_test_r, A_le))

print("\n=== Fairness Evaluation: Residual-Parity Weighted Regression ===")
print(regression_mape_by_group(y_test_r, rsd_weighted.predict(X_test_r), A_test_r, A_le))
